# Lab 7: Writing Data in MongoDB

Santiago Elí Jiménez Aguilar  
Luis Eduardo Gonzalez Gloria

## Goal:
To create a data pipeline to analyze a synthetic Recommendation Videogames Dataset.

## Instructions
- **Dataset**. Create a DataFrame with a nested structure.
- **Enrich Dataset**. Enrich users with their rating history as an array.
- **Writing Data in MongoDB**. This section should contain code to persist the enriched dataset in MongoDB.
- **Inspect MongoDB Collections**. This section should contain a screenshot of documents obtained by inspecting the collection in the MongoDB container.

## Dataset

### Create SparkSession

In [ ]:
from spark_utils import SparkUtils
import pyspark.sql.functions as F

mongodb_connector = "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0"
su = SparkUtils("Lab07: Writing Data in MongoDB",
                "spark://spark-master:7077",
                spark_packages=mongodb_connector)
su.spark

### Videogames DataFrame

In [ ]:
games_schema = SparkUtils.generate_schema([
    ("title",    "string"),
    ("platform", "string"),
    ("genre",    "string"),
    ("rating",   "double"),
    ("reviews",  "int"),
    ("tags",     "array_string"),
    ("publisher", "struct", [
        ("name",    "string"),
        ("country", "string"),
    ]),
])

games_data = [
    ("The Legend of Zelda: Tears of the Kingdom", "Switch", "Action-Adventure",
     4.9, 5100, ["adventure", "RPG", "open-world"], ("Nintendo", "Japan")),
    ("God of War: Ragnarök",                      "PS5",    "Action",
     4.8, 4200, ["action", "adventure", "mythology"], ("Sony Santa Monica", "USA")),
    ("Halo Infinite",                              "Xbox",   "FPS",
     4.3, 2100, ["FPS", "multiplayer", "sci-fi"], ("Microsoft", "USA")),
    ("Stardew Valley",                             "PC",     "Simulation",
     4.7, 6800, ["simulation", "indie", "farming"], ("ConcernedApe", "USA")),
    ("Elden Ring",                                 "PC",     "Action-RPG",
     4.9, 7300, ["RPG", "open-world", "souls-like"], ("FromSoftware", "Japan")),
    ("Hollow Knight",                              "Switch", "Metroidvania",
     4.6, 3900, ["indie", "platformer", "exploration"], ("Team Cherry", "Australia")),
]

games_df = su.spark.createDataFrame(games_data, games_schema)
games_df.show(truncate=False)
games_df.printSchema()

### Users DataFrame

In [ ]:
user_schema = SparkUtils.generate_schema([
    ("user_id",         "string"),
    ("name",            "string"),
    ("email",           "string"),
    ("country",         "string"),
    ("signup_date_str", "string"),
])

users_data = [
    ("u001", "Alice",   "alice@mail.com",   "Mexico",  "2024-01-15"),
    ("u002", "Bob",     "bob@mail.com",     "USA",     "2024-02-20"),
    ("u003", "Carol",   "carol@mail.com",   "Canada",  "2023-11-05"),
    ("u004", "David",   "david@mail.com",   "Spain",   "2024-03-10"),
    ("u005", "Eva",     "eva@mail.com",     "Mexico",  "2024-04-22"),
]

users_df = su.spark.createDataFrame(users_data, user_schema)
users_df = users_df.withColumn(
    "signup_date", F.to_date("signup_date_str", "yyyy-MM-dd")
).drop("signup_date_str")

users_df.show(truncate=False)

### Ratings DataFrame

In [ ]:
ratings_schema = SparkUtils.generate_schema([
    ("user_id",    "string"),
    ("game_title", "string"),
    ("score",      "float"),
    ("date_str",   "string"),
])

ratings_data = [
    ("u001", "The Legend of Zelda: Tears of the Kingdom", 5.0, "2024-06-01"),
    ("u001", "Stardew Valley",                            4.5, "2024-06-15"),
    ("u001", "Hollow Knight",                             4.8, "2024-07-10"),
    ("u002", "God of War: Ragnarök",                      4.8, "2024-07-01"),
    ("u002", "Elden Ring",                                5.0, "2024-07-20"),
    ("u003", "Halo Infinite",                             3.9, "2024-05-20"),
    ("u003", "The Legend of Zelda: Tears of the Kingdom", 4.7, "2024-05-25"),
    ("u003", "Elden Ring",                                4.6, "2024-06-30"),
    ("u004", "Stardew Valley",                            5.0, "2024-08-01"),
    ("u004", "Hollow Knight",                             4.3, "2024-08-15"),
    ("u005", "God of War: Ragnarök",                      4.9, "2024-09-01"),
    ("u005", "Elden Ring",                                5.0, "2024-09-10"),
]

ratings_df = su.spark.createDataFrame(ratings_data, ratings_schema)
ratings_df = ratings_df.withColumn(
    "rate_date", F.to_date("date_str", "yyyy-MM-dd")
).drop("date_str")

ratings_df.show(truncate=False)

## Transformations and Aggregations

In [ ]:
# Aggregate statistics per game
game_stats = (ratings_df
    .groupBy("game_title")
    .agg(
        F.round(F.avg("score"), 2).alias("avg_score"),
        F.count("*").alias("num_ratings"),
        F.max("rate_date").alias("last_rated"),
    )
    .orderBy(F.desc("avg_score")))

game_stats.show(truncate=False)

# Enrich users with their rating history as an array
user_ratings = (ratings_df
    .groupBy("user_id")
    .agg(
        F.collect_list(
            F.struct("game_title", "score", "rate_date")
        ).alias("ratings")
    ))

enriched_users = users_df.join(user_ratings, on="user_id", how="left")
enriched_users.printSchema()
enriched_users.show(truncate=False)

## Write to MongoDB

In [ ]:
mongo_uri = "mongodb://host.docker.internal:27017"

(enriched_users.write
    .format("mongodb")
    .option("database", "videogames")
    .option("collection", "user_ratings")
    .option("connection.uri", mongo_uri)
    .mode("overwrite")
    .save())

print("Data successfully written to MongoDB.")

### Inspecting MongoDB Collections in Docker
```bash
# 1. Open a shell inside the container
docker exec -it mongodb-iteso mongosh
```
```javascript
// 2. List all databases
show dbs

// 3. Switch to the videogames database
use videogames

// 4. List all collections
show collections

// 5. Count documents in the collection
db.user_ratings.countDocuments()

// 6. Preview documents
db.user_ratings.find().limit(5).pretty()
```

> **Screenshot of the MongoDB collection inspection goes here.**

In [ ]:
su.spark.stop()